### Set explicit data directory and load pyspark

In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("data-modeling-kimball-scd2")
    .getOrCreate()
)

print(spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/04 21:56:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/04 21:56:11 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/10/04 21:56:11 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/10/04 21:56:11 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


3.5.4


### SCD2 Scenario - Changing customer attribute

In [3]:
dim_customer = spark.read.parquet(
    str(DATA_DIR / "kimball" / "dim_customer")
)

dim_customer.orderBy("customer_key").show()

+------------+-----------+----------+---------+--------------------+-----------+-----+-------------------+
|customer_key|customer_id|first_name|last_name|               email|       city|state|         created_at|
+------------+-----------+----------+---------+--------------------+-----------+-----+-------------------+
|           1|       1001|     Alice|   Nguyen|alice.nguyen@exam...|Los Angeles|   CA|2026-01-15 09:30:00|
|           2|       1002|    Marcus|     Hill|marcus.hill@examp...|     Austin|   TX|2026-02-03 14:10:00|
|           3|       1003|     Priya|     Shah|priya.shah@exampl...|    Seattle|   WA|2026-02-20 11:45:00|
|           4|       1004|    Daniel|      Kim|daniel.kim@exampl...|    Chicago|   IL|2026-03-08 16:20:00|
|           5|       1005|     Sofia| Martinez|sofia.martinez@ex...|     Denver|   CO|2026-04-01 10:00:00|
+------------+-----------+----------+---------+--------------------+-----------+-----+-------------------+



In [4]:
dim_customer_scd2 = (
    dim_customer
    .withColumn("effective_from", F.lit("2026-01-01").cast("date"))
    .withColumn("effective_to", F.lit(None).cast("date"))
    .withColumn("is_current", F.lit(True))
)

dim_customer_scd2.orderBy("customer_key").show()

+------------+-----------+----------+---------+--------------------+-----------+-----+-------------------+--------------+------------+----------+
|customer_key|customer_id|first_name|last_name|               email|       city|state|         created_at|effective_from|effective_to|is_current|
+------------+-----------+----------+---------+--------------------+-----------+-----+-------------------+--------------+------------+----------+
|           1|       1001|     Alice|   Nguyen|alice.nguyen@exam...|Los Angeles|   CA|2026-01-15 09:30:00|    2026-01-01|        NULL|      true|
|           2|       1002|    Marcus|     Hill|marcus.hill@examp...|     Austin|   TX|2026-02-03 14:10:00|    2026-01-01|        NULL|      true|
|           3|       1003|     Priya|     Shah|priya.shah@exampl...|    Seattle|   WA|2026-02-20 11:45:00|    2026-01-01|        NULL|      true|
|           4|       1004|    Daniel|      Kim|daniel.kim@exampl...|    Chicago|   IL|2026-03-08 16:20:00|    2026-01-01|   

In [5]:
bronze_customers = spark.read.parquet(
    str(DATA_DIR / "bronze" / "crm_customers")
)

current_dim = dim_customer_scd2.filter(F.col("is_current") == True)

comparison = (
    bronze_customers.alias("b")
    .join(
        current_dim.alias("d"),
        F.col("b.customer_id") == F.col("d.customer_id"),
        "left"
    )
)

changed_customers = (
    comparison
    .filter(
        (F.col("d.customer_id").isNull()) |
        (F.col("b.first_name") != F.col("d.first_name")) |
        (F.col("b.last_name") != F.col("d.last_name")) |
        (F.col("b.email") != F.col("d.email")) |
        (F.col("b.city") != F.col("d.city")) |
        (F.col("b.state") != F.col("d.state"))
    )
)

changed_customers.select(
    F.col("b.customer_id").alias("customer_id"),
    F.col("d.customer_key").alias("existing_customer_key"),
    F.col("d.city").alias("old_city"),
    F.col("b.city").alias("new_city")
).show()

+-----------+---------------------+-----------+---------+
|customer_id|existing_customer_key|   old_city| new_city|
+-----------+---------------------+-----------+---------+
|       1001|                    1|Los Angeles|San Diego|
+-----------+---------------------+-----------+---------+



In [6]:
existing_changes = changed_customers.filter(
    F.col("d.customer_key").isNotNull()
)

existing_changes.select(
    F.col("b.customer_id").alias("customer_id"),
    F.col("d.customer_key").alias("customer_key")
).show()

changed_keys = (
    existing_changes
    .select(F.col("d.customer_key").alias("customer_key"))
    .distinct()
)

change_date = "2026-10-04"

expired_dim = (
    dim_customer_scd2.alias("d")
    .join(
        changed_keys.alias("c"),
        on="customer_key",
        how="left"
    )
    .withColumn(
        "effective_to",
        F.when(
            F.col("c.customer_key").isNotNull() & F.col("d.is_current"),
            F.date_sub(F.lit(change_date).cast("date"), 1)
        ).otherwise(F.col("d.effective_to"))
    )
    .withColumn(
        "is_current",
        F.when(
            F.col("c.customer_key").isNotNull() & F.col("d.is_current"),
            F.lit(False)
        ).otherwise(F.col("d.is_current"))
    )
    .select("d.*", "effective_to", "is_current")
)

+-----------+------------+
|customer_id|customer_key|
+-----------+------------+
|       1001|           1|
+-----------+------------+



In [7]:
expired_dim = (
    dim_customer_scd2
    .join(changed_keys, on="customer_key", how="left_semi")
)

In [8]:
change_date = "2026-10-04"

expired_rows = (
    existing_changes
    .select(
        F.col("d.customer_key").alias("customer_key"),
        F.col("d.customer_id").alias("customer_id"),
        F.col("d.first_name").alias("first_name"),
        F.col("d.last_name").alias("last_name"),
        F.col("d.email").alias("email"),
        F.col("d.city").alias("city"),
        F.col("d.state").alias("state"),
        F.col("d.created_at").alias("created_at"),
        F.col("d.effective_from").alias("effective_from"),
        F.date_sub(
            F.lit(change_date).cast("date"), 1
        ).alias("effective_to"),
        F.lit(False).alias("is_current")
    )
)

expired_rows.show()

+------------+-----------+----------+---------+--------------------+-----------+-----+-------------------+--------------+------------+----------+
|customer_key|customer_id|first_name|last_name|               email|       city|state|         created_at|effective_from|effective_to|is_current|
+------------+-----------+----------+---------+--------------------+-----------+-----+-------------------+--------------+------------+----------+
|           1|       1001|     Alice|   Nguyen|alice.nguyen@exam...|Los Angeles|   CA|2026-01-15 09:30:00|    2026-01-01|  2026-10-03|     false|
+------------+-----------+----------+---------+--------------------+-----------+-----+-------------------+--------------+------------+----------+



In [9]:
max_customer_key = (
    dim_customer_scd2
    .agg(F.max("customer_key").alias("max_key"))
    .collect()[0]["max_key"]
)

print(max_customer_key)

5


In [10]:
new_current_rows = (
    existing_changes
    .select(
        F.lit(max_customer_key + 1).alias("customer_key"),
        F.col("b.customer_id").alias("customer_id"),
        F.col("b.first_name").alias("first_name"),
        F.col("b.last_name").alias("last_name"),
        F.col("b.email").alias("email"),
        F.col("b.city").alias("city"),
        F.col("b.state").alias("state"),
        F.col("b.created_at").alias("created_at"),
        F.lit(change_date).cast("date").alias("effective_from"),
        F.lit(None).cast("date").alias("effective_to"),
        F.lit(True).alias("is_current")
    )
)

new_current_rows.show()

+------------+-----------+----------+---------+--------------------+---------+-----+-------------------+--------------+------------+----------+
|customer_key|customer_id|first_name|last_name|               email|     city|state|         created_at|effective_from|effective_to|is_current|
+------------+-----------+----------+---------+--------------------+---------+-----+-------------------+--------------+------------+----------+
|           6|       1001|     Alice|   Nguyen|alice.nguyen@exam...|San Diego|   CA|2026-01-15 09:30:00|    2026-10-04|        NULL|      true|
+------------+-----------+----------+---------+--------------------+---------+-----+-------------------+--------------+------------+----------+



In [11]:
unchanged_rows = (
    dim_customer_scd2
    .join(
        changed_keys,
        on="customer_key",
        how="left_anti"
    )
)

final_dim_customer_scd2 = (
    unchanged_rows
    .unionByName(expired_rows)
    .unionByName(new_current_rows)
    .cache()
)

final_dim_customer_scd2.count()

6

In [12]:

final_dim_customer_scd2.filter(
    F.col("customer_id") == 1001
).orderBy("effective_from").show()

+------------+-----------+----------+---------+--------------------+-----------+-----+-------------------+--------------+------------+----------+
|customer_key|customer_id|first_name|last_name|               email|       city|state|         created_at|effective_from|effective_to|is_current|
+------------+-----------+----------+---------+--------------------+-----------+-----+-------------------+--------------+------------+----------+
|           1|       1001|     Alice|   Nguyen|alice.nguyen@exam...|Los Angeles|   CA|2026-01-15 09:30:00|    2026-01-01|  2026-10-03|     false|
|           6|       1001|     Alice|   Nguyen|alice.nguyen@exam...|  San Diego|   CA|2026-01-15 09:30:00|    2026-10-04|        NULL|      true|
+------------+-----------+----------+---------+--------------------+-----------+-----+-------------------+--------------+------------+----------+



In [13]:
final_dim_customer_scd2.write.mode("overwrite").parquet(
    str(DATA_DIR / "kimball" / "dim_customer")
)

In [14]:
dim_customer_scd2_reloaded = spark.read.parquet(
    str(DATA_DIR / "kimball" / "dim_customer")
)

In [15]:
dim_customer_scd2_reloaded.show()

+------------+-----------+----------+---------+--------------------+-----------+-----+-------------------+--------------+------------+----------+
|customer_key|customer_id|first_name|last_name|               email|       city|state|         created_at|effective_from|effective_to|is_current|
+------------+-----------+----------+---------+--------------------+-----------+-----+-------------------+--------------+------------+----------+
|           1|       1001|     Alice|   Nguyen|alice.nguyen@exam...|Los Angeles|   CA|2026-01-15 09:30:00|    2026-01-01|  2026-10-03|     false|
|           5|       1005|     Sofia| Martinez|sofia.martinez@ex...|     Denver|   CO|2026-04-01 10:00:00|    2026-01-01|        NULL|      true|
|           6|       1001|     Alice|   Nguyen|alice.nguyen@exam...|  San Diego|   CA|2026-01-15 09:30:00|    2026-10-04|        NULL|      true|
|           2|       1002|    Marcus|     Hill|marcus.hill@examp...|     Austin|   TX|2026-02-03 14:10:00|    2026-01-01|   